# Comparing BattleActs Exam Preparation Samples
**DATASCI 530 — Project 2 | Maxine Hakimi-Bulson**

This project compares vocabulary, text lengths, and practice formats in the public samples for CAS Exams 5, 6C, 6U, 7, 8, and 9. Materials with the same purpose are compared across exams. Product descriptions provide a brief supporting comparison.

## Key findings — complete after analysis
Replace the following prompts with **3–4 numbered findings supported by actual results** before submission:
1. [A major vocabulary difference, including a number and figure reference.]
2. [A vocabulary similarity, including a number and comparison group.]
3. [A reliable question-format or sample-length finding.]
4. [An optional product-description finding.]

Public samples do not establish the quality or coverage of full courses.

## Start here
This is a working analysis notebook, not a completed report. Run cells in order and complete the interpretation prompts.

1. Put this notebook beside the three Project 1 CSVs. Adjust the paths below.
2. Keep your original downloads. The file manifest will let you point to them without scraping again.
3. Check extraction, material categories, and exclusions before interpreting results.
4. Review and edit the code for your project; remove unused sections and draft prompts.
5. Export the final notebook to HTML and submit its associated CSV datasets.

**Dependencies:** pandas, numpy, matplotlib, scikit-learn, pypdf, openpyxl, and optionally wordcloud. If needed, install in the notebook's Python environment using `%pip install pandas numpy matplotlib scikit-learn pypdf openpyxl wordcloud`.

The uploaded Project 1 notebook was unavailable, so this notebook imports saved datasets independently rather than modifying your scraper.

## Rubric map
Verified against **DATASCI530_Projects 1 and 2 Instructions.pdf**, Project 2 rubric, page 3.

| Criterion | Points | Where addressed |
|---|---:|---|
| Code organization/documentation and aesthetics | 3 | Short method cells, reusable functions, formatted outputs |
| Description and 3–4 opening findings | 2 | Introduction and findings at top |
| Address Project 1 feedback | 4 | Feedback response, extraction review, export checks |
| Tables with observations and summary statistics | 4 | Inventory, quality checks, descriptive summaries |
| Professional figures with interpretation | 6 | Frequency, similarity, length and optional format figures |
| Clear, concise discussion | 1 | Discussion and limitations |

Source: [Assignment instructions](https://chatgpt.com/library?file_id=libfile_484f40c89b008191b292f9c49718cbff&account_id=personal).
No template can guarantee a grade. The final findings, validation, and interpretations must be completed.

## 1. Import Project 1 datasets
This separate notebook imports the original CSVs as required. It makes no network requests and does not change collection dates or overwrite source files. The optional Products file is needed for the product-description section.

In [ ]:
from pathlib import Path
from collections import Counter
import re, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

BASE = Path.cwd()
PDF_CSV = BASE / 'battleacts_pdf_samples.csv'
EXCEL_CSV = BASE / 'battleacts_excel_samples.csv'
PRODUCTS_CSV = BASE / 'battleacts_products.csv'
OUT = BASE / 'project2_outputs'
OUT.mkdir(exist_ok=True)
for path in [PDF_CSV, EXCEL_CSV]:
    if not path.exists():
        raise FileNotFoundError(f'Update the path in this cell: {path}')
pdf = pd.read_csv(PDF_CSV)
excel = pd.read_csv(EXCEL_CSV)
products = pd.read_csv(PRODUCTS_CSV) if PRODUCTS_CSV.exists() else pd.DataFrame()
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11, 'axes.spines.top': False,
                     'axes.spines.right': False, 'figure.constrained_layout.use': True})
def show_table(frame, caption, precision=2):
    display(frame.style.format(precision=precision, na_rep='Unavailable')
            .set_caption(caption).hide(axis='index')
            .set_properties(**{'text-align': 'center'})
            .set_table_styles([{'selector': 'th', 'props': [('text-align','center')]}]))
frames=[]
for frame, kind in [(pdf,'PDF'), (excel,'Excel')]:
    f=frame.copy()
    f['file_type']=kind
    f['source_csv']=PDF_CSV.name if kind=='PDF' else EXCEL_CSV.name
    frames.append(f)
files=pd.concat(frames, ignore_index=True)
files['file_id']=[f'F{i:02}' for i in range(1,len(files)+1)]
assert not files['sample_url'].duplicated().any(), 'Investigate repeated file URLs before proceeding.'
show_table(files.groupby(['product_name','file_type']).size().rename('Files').reset_index(),
           'Table 1. Unique source files by exam and file type', 0)

## 2. Address Project 1 feedback and review the manifest
The CSV display differed between Numbers and the professor's Excel. Successful CSV parsing alone does not establish extraction quality. Re-extract from saved originals, check representative content, and avoid storing whole books in one spreadsheet cell. Formula objects require explicit handling.

The first run creates `file_manifest.csv`. Edit its **local_path**, **material_category**, and **reviewed** columns, save it, and rerun this cell. Use absolute paths or paths relative to this notebook. Inspect content rather than relying only on button labels. Valid categories: `study_manual`, `formula_summary`, `practice_material`, `other`. A merged label may point to one file, not two materials.

Record what you actually checked in the feedback log below; planned corrections are not completed corrections.

In [ ]:
manifest_path=BASE / 'file_manifest.csv'
if manifest_path.exists():
    manifest=pd.read_csv(manifest_path).fillna('')
    assert set(manifest.file_id)==set(files.file_id), 'Manifest does not match imported files.'
else:
    manifest=files[['file_id','product_name','sample_label','sample_url','file_type']].copy()
    manifest['local_path']=''
    manifest['material_category']=''
    manifest['reviewed']='no'
    manifest.to_csv(manifest_path,index=False)
show_table(manifest, 'Table 2. File inventory — edit the manifest before final analysis',0)

feedback_log=pd.DataFrame([
    ['Excel display issue','Re-export smaller text units; verify in Excel','Pending'],
    ['Formula object text','Extract formula expressions separately from prose','Pending'],
    ['Extraction validation','Compare source passages and extracted text for every exam','Pending'],
    ['Text-based analysis','Add frequencies, vocabulary overlap and interpretations','Pending']
], columns=['Feedback','Action','Verified status'])
show_table(feedback_log,'Project 1 feedback response — update statuses with evidence')

## 3. Re-extract saved originals
PDFs are extracted by page. Excel prose is extracted by row within each worksheet; formulas are preserved separately. This does not calculate formulas or recover text embedded in images. Coordinates are retained for checks. Files lacking a local path are reported as pending, rather than silently treated as empty.

Do not proceed to final comparisons until all intended files have reviewed categories and successful extraction.

In [ ]:
from pypdf import PdfReader
from openpyxl import load_workbook

units, formula_records, extraction_log = [], [], []
def add_unit(meta, unit, text):
    units.append({**meta, 'unit':unit, 'text':text})
for _, row in manifest.iterrows():
    meta={k:row[k] for k in ['file_id','product_name','sample_url','material_category']}
    dates=files.loc[files.file_id.eq(row.file_id)]
    for col in ['collection_date','content_collection_date']:
        if col in dates: meta[col]=dates.iloc[0][col]
    path=Path(row.local_path).expanduser() if row.local_path else None
    if path is not None and not path.is_absolute(): path=BASE/path
    if path is None or not path.exists():
        extraction_log.append([row.file_id,'Pending: original file path needed']); continue
    try:
        if row.file_type=='PDF':
            reader=PdfReader(path)
            for number,page in enumerate(reader.pages,1):
                add_unit(meta,f'Page {number}',page.extract_text() or '')
        else:
            book=load_workbook(path,data_only=False,read_only=False)
            for sheet in book:
                for cells in sheet.iter_rows():
                    parts=[]
                    for cell in cells:
                        value=cell.value
                        if value is None: continue
                        if cell.data_type=='f':
                            expression=value if isinstance(value,str) else getattr(value,'text',None)
                            formula_records.append({**meta,'sheet':sheet.title,'cell':cell.coordinate,
                                'formula':expression, 'formula_object_type':type(value).__name__})
                        elif isinstance(value,str):
                            parts.append(f'{cell.coordinate}: {value}')
                    if parts: add_unit(meta,f'{sheet.title} / row {cells[0].row}', '\n'.join(parts))
            book.close()
        extraction_log.append([row.file_id,'Extracted — content review still required'])
    except Exception as error:
        extraction_log.append([row.file_id,f'FAILED: {type(error).__name__}: {error}'])
text_units=pd.DataFrame(units,columns=['file_id','product_name','sample_url','material_category',
                                     'collection_date','content_collection_date','unit','text'])
formula_data=pd.DataFrame(formula_records)
show_table(pd.DataFrame(extraction_log,columns=['File','Status']),'Extraction log')
if len(text_units):
    show_table(text_units.assign(characters=text_units.text.str.len())
               .groupby('file_id').agg(Units=('unit','size'),Characters=('characters','sum')).reset_index(),
               'Table 3. Extracted units and character counts',0)

### Manual extraction checks
For every exam, compare a representative PDF page and worksheet with the original where available. Check the beginning and end, formulas or tables, and unusual layouts. Inspect blank pages and unusually long units. Add rows to this log as needed. Complete source locations, findings, and verification status; do not mark a check complete without doing it.

In [ ]:
check_path=BASE/'extraction_review.csv'
if check_path.exists():
    review=pd.read_csv(check_path).fillna('')
else:
    review=pd.DataFrame(columns=['file_id','source_location','checked_content','finding','verified'])
    review.to_csv(check_path,index=False)
show_table(review,'Extraction review log')

## 4. Clean text and define comparable groups
Only reviewed materials with assigned categories enter analysis. Keep raw text unchanged. Exclude instruction/contents sheets and repeated administrative text based on source review. Edit the exclusion list below to match the actual files. Two-word phrases are formed within lines before stopword filtering; they never span pages, worksheets, or rows. Formula symbols and expressions are outside the vocabulary analysis.

Inspect the leading terms and original passages before interpreting themes. A frequent word is not necessarily an important topic.

In [ ]:
EXCLUDE_SHEETS={'instructions','point grid','toc','contents','table of contents'}
EXCLUDE_LINE_PATTERNS=[r'return to point grid',r'^incomplete$',r'^slay the beast$']
EXTRA_STOPWORDS=set()  # Add only justified boilerplate terms and explain changes.
STOP=set(ENGLISH_STOP_WORDS)|EXTRA_STOPWORDS
valid_categories={'study_manual','formula_summary','practice_material','other'}
def cleaned_lines(row):
    unit=row['unit']
    if ' / row ' in unit and unit.split(' / row ')[0].strip().lower() in EXCLUDE_SHEETS:
        return []
    lines=[]
    for line in str(row['text']).splitlines():
        line=re.sub(r'^[A-Z]+[0-9]+:\s*','',line).lower().strip()
        if not any(re.search(p,line) for p in EXCLUDE_LINE_PATTERNS): lines.append(line)
    return lines
def counts_from_lines(lines):
    words,phrases=Counter(),Counter()
    for line in lines:
        tokens=re.findall(r"[a-z]+(?:'[a-z]+)?",line)
        words.update(t for t in tokens if t not in STOP and len(t)>1)
        phrases.update(' '.join(pair) for pair in zip(tokens,tokens[1:])
                       if all(t not in STOP and len(t)>1 for t in pair))
    return words,phrases

if text_units.empty:
    raise ValueError('Supply saved original file paths in file_manifest.csv and rerun extraction.')
approved=set(manifest.loc[manifest.reviewed.astype(str).str.lower().eq('yes') &
    manifest.material_category.isin(valid_categories),'file_id'])
analysis_units=text_units[text_units.file_id.isin(approved)].copy()
if analysis_units.empty:
    raise ValueError('Review file contents, assign categories, and set reviewed=yes in the manifest.')
documents=[]
for (exam,category),group in analysis_units.groupby(['product_name','material_category']):
    lines=[line for _,r in group.iterrows() for line in cleaned_lines(r)]
    words,phrases=counts_from_lines(lines)
    denominator=sum(len(re.findall(r"[a-z]+(?:'[a-z]+)?",line)) for line in lines)
    documents.append({'exam':exam,'category':category,'files':group.file_id.nunique(),
        'word_count':denominator,'retained_words':sum(words.values()),'words':words,'phrases':phrases})
docs=pd.DataFrame(documents)
show_table(docs.drop(columns=['words','phrases']),
           'Table 4. Available text by exam and material category',0)
summary=docs.groupby('category').agg(Exam_groups=('exam','size'),
    Total_words=('word_count','sum'),Mean_words=('word_count','mean'),
    Variance_words=('word_count','var')).reset_index()
show_table(summary,'Table 5. Summary statistics across available exam groups')

## 5. Frequent terms and phrases
Frequencies use retained matches per 1,000 prose words before stopword removal. The denominator excludes discarded administrative lines. Review the raw count and denominator alongside each rate. Select an available material category below. Missing categories are not zeros.

In [ ]:
CATEGORY='study_manual'  # Change to formula_summary or practice_material as appropriate.
selected=docs[docs.category.eq(CATEGORY)]
frequency_rows=[]
for _,row in docs.iterrows():
    for kind in ['words','phrases']:
        for term,count in row[kind].items():
            frequency_rows.append([row.exam,row.category,kind,term,count,row.word_count,
                1000*count/row.word_count if row.word_count else np.nan])
frequencies=pd.DataFrame(frequency_rows,columns=['Exam','Category','Kind','Term','Count',
                                              'Prose words','Per 1000 words'])
top=frequencies.query('Category == @CATEGORY and Kind == "phrases"')
if top.empty:
    print('No phrases for this category. Choose another available category.')
else:
    leaders=top.groupby('Term')['Per 1000 words'].mean().nlargest(8).index
    comparison=top[top.Term.isin(leaders)].pivot(index='Term',columns='Exam',values='Per 1000 words')
    comparison=comparison.reindex(columns=selected.exam).fillna(0)
    ax=comparison.plot.barh(figsize=(10,6),width=.8)
    ax.set(xlabel='Occurrences per 1,000 prose words',ylabel='Phrase',
           title=f'Figure 1. Frequent phrases: {CATEGORY.replace("_"," ")}')
    ax.legend(title='Exam',bbox_to_anchor=(1.02,1),loc='upper left')
    plt.savefig(OUT/f'phrase_frequencies_{CATEGORY}.png',bbox_inches='tight');plt.show()
    show_table(top.sort_values('Per 1000 words',ascending=False).groupby('Exam').head(5),
               'Table 6. Leading phrases by exam')

**Interpret Figure 1:** Which phrases differ most? Give at least two numerical comparisons. Read their original contexts. Could boilerplate, sample length, or material selection explain the differences?

In [ ]:
# Retrieve passages before interpreting a frequent term.
TERM='loss adjustment'
matches=analysis_units[analysis_units.text.str.contains(re.escape(TERM),case=False,na=False)]
for _,row in matches.head(5).iterrows():
    match=re.search(re.escape(TERM),row.text,re.I)
    excerpt=row.text[max(0,match.start()-100):match.end()+150]
    print(row.product_name,row.file_id,row.unit,'\n',excerpt,'\n')

### Word clouds — optional overview
Larger words indicate higher within-sample frequency, not importance or syllabus coverage. Use bar charts for numerical comparisons. Omit this section if the category has too little readable text.

In [ ]:
try:
    from wordcloud import WordCloud
    available=selected[selected.retained_words.gt(0)]
    if len(available):
        fig,axes=plt.subplots(len(available),1,figsize=(9,3*len(available)),squeeze=False)
        for ax,(_,row) in zip(axes.flat,available.iterrows()):
            cloud=WordCloud(width=900,height=280,background_color='white',
                            colormap='viridis',random_state=530).generate_from_frequencies(row.words)
            ax.imshow(cloud,interpolation='bilinear');ax.axis('off');ax.set_title(row.exam)
        fig.suptitle(f'Figure 2. Frequent vocabulary: {CATEGORY.replace("_"," ")}')
        plt.savefig(OUT/f'wordclouds_{CATEGORY}.png',bbox_inches='tight');plt.show()
except ImportError:
    print('Optional wordcloud package unavailable; frequency analysis is still available.')

## 6. Vocabulary overlap
Within the selected category, Jaccard similarity is the number of distinct retained words shared by two exam groups divided by the number in their union. It ranges from 0 to 1. Different sample lengths and remaining boilerplate affect this measure. It does not measure conceptual equivalence or difficulty.

In [ ]:
similarity_rows=[]
if len(selected)>=2:
    names=selected.exam.tolist();sets=[set(w) for w in selected.words]
    matrix=np.full((len(sets),len(sets)),np.nan)
    for i,a in enumerate(sets):
        for j,b in enumerate(sets):
            matrix[i,j]=len(a&b)/len(a|b) if a|b else np.nan
            if i<j: similarity_rows.append([CATEGORY,names[i],names[j],matrix[i,j]])
    fig,ax=plt.subplots(figsize=(8,6))
    image=ax.imshow(matrix,vmin=0,vmax=1,cmap='Blues')
    ax.set_xticks(range(len(names)),names,rotation=45,ha='right')
    ax.set_yticks(range(len(names)),names)
    for i in range(len(names)):
        for j in range(len(names)):
            ax.text(j,i,f'{matrix[i,j]:.2f}',ha='center',va='center',
                    color='white' if matrix[i,j]>.6 else 'black')
    ax.set_title(f'Figure 3. Vocabulary overlap: {CATEGORY.replace("_"," ")}')
    fig.colorbar(image,ax=ax,label='Jaccard similarity')
    plt.savefig(OUT/f'vocabulary_overlap_{CATEGORY}.png',bbox_inches='tight');plt.show()
else:
    print('At least two exams with this material category are needed.')
similarities=pd.DataFrame(similarity_rows,columns=['Category','Exam A','Exam B','Jaccard'])

**Interpret Figure 3:** Identify the highest and lowest off-diagonal similarities, quote their values, and discuss source passages and sample lengths. Repeat the category-specific cells for other comparable materials if useful; retain only informative figures in the final report.

## 7. Practice-question formats — reviewed extension
Worksheet counts are not question counts. Different workbooks require different extraction rules. Create `questions_reviewed.csv` after reviewing each workbook, with one row per main question. Include `file_id`, `question_id`, `format`, `subparts`, and `verified`. Valid formats: multiple_choice, numerical, written, mixed, unclear. Exclude navigation sheets and avoid counting a question again in its solution. Mark verified=yes only after checking the source. If reliable identification is not possible, omit this analysis and explain why.

In [ ]:
question_path=BASE/'questions_reviewed.csv'
if question_path.exists():
    questions=pd.read_csv(question_path)
    required={'file_id','question_id','format','subparts','verified'}
    assert required.issubset(questions.columns)
    assert not questions.duplicated(['file_id','question_id']).any()
    assert questions['format'].isin(['multiple_choice','numerical','written','mixed','unclear']).all()
    assert questions.file_id.isin(files.file_id).all()
    questions=questions[questions.verified.astype(str).str.lower().eq('yes')].merge(
        files[['file_id','product_name']],on='file_id',validate='many_to_one')
    if len(questions):
        format_counts=pd.crosstab(questions.product_name,questions['format'])
        format_pct=format_counts.div(format_counts.sum(axis=1),axis=0)*100
        show_table(format_counts.reset_index(),'Table 7. Verified main questions by format',0)
        ax=format_pct.plot.barh(stacked=True,figsize=(9,5),colormap='tab10')
        ax.set(xlabel='Percentage of verified main questions',ylabel='Exam',xlim=(0,100),
               title='Figure 4. Formats in reviewed practice samples')
        ax.legend(title='Format',bbox_to_anchor=(1.02,1));plt.show()
else:
    questions=pd.DataFrame()
    print('Optional extension pending source review; no question-format claims are made.')

## 8. Product descriptions — supporting analysis
Descriptions remain separate from feature lists. Examine length and repeated wording across six courses. High overlap may indicate a shared advertising template. It does not establish course similarity.

In [ ]:
if len(products) and {'description','product_name'}.issubset(products):
    description_rows=[]
    for _,r in products.iterrows():
        description=str(r.description) if pd.notna(r.description) else ''
        words,_=counts_from_lines(description.lower().splitlines())
        description_rows.append([r.product_name,len(re.findall(r"[a-z]+",description.lower())),
                                 ', '.join(t for t,_ in words.most_common(5))])
    description_summary=pd.DataFrame(description_rows,columns=['Exam','Description words','Leading terms'])
    show_table(description_summary,'Table 8. Product-description lengths and frequent terms',0)
else:
    print('Supply the saved Products CSV to complete this section.')

## 9. Discussion
**Write after running the analysis:** Summarize 3–4 supported findings and explain what readers learn from them. Refer to specific tables/figures and numerical values. Distinguish vocabulary frequency from topic importance. Discuss sample purpose, length, extraction limitations, small file counts, and unmatched material categories. Explain that absence in a sample does not establish absence from the full course.

### Feedback response — finalize with evidence
State exactly how you revised the PDF/Excel datasets, what comparisons against originals you performed, and how you checked exported files. Mention the Excel display discrepancy without claiming that it explains all extraction issues. Record unresolved limitations honestly.

## 10. Export and submission
Export derived data separately from the originals. Long text is split into sequential chunks below for spreadsheet compatibility; chunks are for storage, not independent observations. JSON preserves the complete extraction. Small derived CSVs support the tables and figures. UTF-8 with BOM helps encoding but is not by itself proof of Excel compatibility.

In [ ]:
def spreadsheet_text_chunks(frame,limit=12000):
    records=[]
    for _,row in frame.iterrows():
        text=str(row['text'])
        parts=[text[i:i+limit] for i in range(0,len(text),limit)] or ['']
        for number,part in enumerate(parts,1):
            records.append({**row.to_dict(),'chunk_number':number,'text':part})
        assert ''.join(parts)==text
    return pd.DataFrame(records)
chunked=spreadsheet_text_chunks(text_units)
assert chunked.text.str.len().max()<=12000
exports={'text_units_chunks':chunked,'file_manifest':manifest,
         'text_summary':docs.drop(columns=['words','phrases']),
         'summary_statistics':summary,'term_frequencies':frequencies,
         'vocabulary_similarity':similarities,'extraction_review':review}
if len(formula_data): exports['spreadsheet_formulas']=formula_data
if len(questions): exports['reviewed_questions']=questions
for name,frame in exports.items():
    path=OUT/f'{name}.csv'
    frame.to_csv(path,index=False,encoding='utf-8-sig')
    check=pd.read_csv(path,keep_default_na=False)
    assert len(check)==len(frame) and list(check.columns)==list(frame.columns)
text_units.to_json(OUT/'text_units_full.json',orient='records',force_ascii=False,indent=2)
print(f'Saved {len(exports)} derived CSVs and complete-text JSON in {OUT}')

### Final rubric checklist
- [ ] Separate notebook imports Project 1 datasets.
- [ ] Introduction explains the question and scope.
- [ ] Top of report contains 3–4 actual findings with numbers.
- [ ] Project 1 feedback is addressed with completed actions and evidence.
- [ ] Source text and categories are reviewed for every included exam.
- [ ] Tables show observations and relevant summary statistics, with readable captions and rounding.
- [ ] Figures have titles, axis labels, suitable colors, readable fonts, and legends when needed.
- [ ] Each retained figure/table is interpreted for a non-coder audience.
- [ ] Unrun extensions, duplicate outputs, draft prompts, and unused code are removed.
- [ ] Discussion is concise and does not generalize public samples to full courses.
- [ ] Exported CSVs are checked, including opening in Excel where available.
- [ ] Restart kernel and Run All successfully from your configured files.
- [ ] Export HTML using Jupyter's File → Save and Export Notebook As → HTML (wording varies).
- [ ] Open the HTML and verify figures and tables before submitting HTML and CSV files.

### Sources
- BattleActs public course pages and linked samples: source URLs are retained in the imported data.
- DATASCI 530 Projects 1 and 2 instructions, Project 2 rubric, page 3.
- Disclose AI assistance as required by your course; review and edit the code rather than submitting an unreviewed template.